In [1]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama


import requests
from langchain_community.tools import DuckDuckGoSearchRun
import os
from dotenv import load_dotenv

load_dotenv()


c:\Users\ADITYA KUMAR PANDEY\langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\ADITYA KUMAR PANDEY\AppData\Local\Temp\ipykernel_26032\2928605208.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


True

In [2]:
search_tool = DuckDuckGoSearchRun()

# result = search_tool.invoke("capital of india")

# result

In [3]:
API_KEY = os.getenv("WEATHERSTACK_API_KEY")

@tool
def get_weather(city : str) -> str:
    '''This function give current weather of given city'''
    url = f'http://api.weatherstack.com/current'
    
    params = {
        "access_key": API_KEY,
        "query": city
    }
    response = requests.get(url,params=params)
    
    return response.json()

In [4]:
llm = ChatOllama(
    model="qwen3.5:4b",
    temperature=0
)

In [5]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [6]:
from langchain_core.prompts import PromptTemplate

#creating prompt template due to unable to use prompt from hub

template = """Answer the following question as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: You should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [7]:
#creating agent -> it handle all logical or reasoning 

agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather],
    prompt=prompt
)

In [8]:
#creating agent executor -> it maintain loop or work flow until final answer

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather],
    verbose=True,
    handle_parsing_errors=True
)

In [9]:
import os

print("API Key exists:", bool(os.getenv("OPENROUTER_API_KEY")))

API Key exists: True


In [10]:

query = input("How can I assist You ?")

result = agent_executor.invoke({"input" : query})



print("AI response : ", result['output'])
    



> Entering new AgentExecutor chain...
Thought: I need to find the current weather in Chandigarh, so I should use the get_weather function with the city name.

Action: get_weather
Action Input: Chandigarh{'request': {'type': 'City', 'query': 'Chandigarh, India', 'language': 'en', 'unit': 'm'}, 'location': {'name': 'Chandigarh', 'country': 'India', 'region': 'Chandigarh', 'lat': '30.737', 'lon': '76.787', 'timezone_id': 'Asia/Kolkata', 'localtime': '2026-10-21 22:35', 'localtime_epoch': 1792622100, 'utc_offset': '5.50'}, 'current': {'observation_time': '05:05 PM', 'temperature': 21, 'weather_code': 149, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0006_mist.png'], 'weather_descriptions': ['Smoky haze'], 'astro': {'sunrise': '06:29 AM', 'sunset': '05:46 PM', 'moonrise': '02:55 PM', 'moonset': '01:24 AM', 'moon_phase': 'Waxing Gibbous', 'moon_illumination': 72}, 'air_quality': {'co': '1021', 'no2': '49.3', 'o3': '55', 'so2': '17.1', 'pm2_5': '56.